## Grover'ın Arama Algoritması: Kuantum Hızlandırma

### 1\. Amaç: Sıralanmamış Veritabanında Arama

**Problem:** Elinizde $N$ elemanlı, sıralanmamış bir veritabanı (veya fonksiyon) var. İçinde sadece bir tane "doğru" veya "aranan" eleman ($|w\rangle$) bulunuyor.

  * **Klasik Yaklaşım:** Klasik bir bilgisayar, bu elemanı bulmak için en kötü durumda $N$ sorgulama (elemanı kontrol etme) yapmak zorundadır. Ortalama olarak $N/2$ sorgu gerekir.

  * **Kuantum Yaklaşım (Grover):** Kuantum bilgisayar, yalnızca $\mathcal{O}(\sqrt{N})$ sorgulama yaparak aranan elemanı bulabilir.

Bu, klasik hesaplamaya göre **kuadratik bir hızlanmadır**. Örneğin, $N=1.000.000$ elemanlı bir listede:

  * Klasik: Ortalama $500.000$ sorgu.
  * Grover: $\sqrt{1.000.000} = 1.000$ sorgu.

### 2\. Algoritmanın Temel Mantığı: Genlik Yükseltme

Grover algoritması, aranan duruma ait **olasılık genliğini sistematik olarak yükselterek** çalışır.

Algoritma, basitçe, iki temel adımdan oluşan bir döngüden (Grover İterasyonu) oluşur:

1.  **Kuantum Oracle ($U_w$):** Aranan elemanın genliğine **ters faz** uygular.
2.  **Grover Difüzör ($D$):** Tüm genliklerin ortalaması etrafında bir yansıma gerçekleştirir, bu da aranan elemanın genliğini yükseltirken diğerlerini düşürür.

Bu iki adım, aranan durumun olasılığını her iterasyonda artırır. $\sqrt{N}$ iterasyon sonunda, aranan durumu ölçme olasılığı %99'a yaklaşır.

### 3\. Grover İterasyonunun Bileşenleri

Grover iterasyonunda $N=2^n$ qubit kullanılır.

#### A. Kuantum Oracle ($U_w$)

Oracle'ın görevi, aranan durumun ($|w\rangle$) fazını ($|w\rangle \to -|w\rangle$) çevirmektir.

$$\text{Oracle} (U_w) = I - 2|w\rangle\langle w|$$

  * **Matematiksel Etki:** Aranan durum dışındaki tüm durumları sabit tutar. Aranan duruma bir $\pi$ radyanlık ($-1$) faz ekler.

#### B. Grover Difüzör ($D$)

Difüzör kapısı, tüm durumların ortalama genliği etrafında bir yansıma görevi görür. Bu, aranan durumun genliğini ortalamanın üzerine iterek yükseltir.

$$\text{Difüzör} (D) = 2|\psi_0\rangle\langle \psi_0| - I$$

Burada $|\psi_0\rangle$ başlangıç süperpozisyon durumudur ($\frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} |x\rangle$).

  * **Uygulanması:** Difüzör, basit kapılarla oluşturulabilir:
    $$D = -H^{\otimes n} Z^{\text{kontrol}}_{\text{tüm 0'lar}} H^{\otimes n}$$
      * $H^{\otimes n}$: Tüm qubitlere Hadamard kapısı.
      * $Z^{\text{kontrol}}_{\text{tüm 0'lar}}$: Sadece $|00...0\rangle$ durumuna $Z$ fazı uygulayan bir kapı.

### 4\. Cirq Kod Örneği: $N=4$ Durum Arasında Arama

$N=4$ olası durum için ($|00\rangle, |01\rangle, |10\rangle, |11\rangle$), $n=2$ qubit kullanılır.

**Aranan durum:** $|11\rangle$ (Yani çıktı 3'ü arıyoruz).

$\sqrt{N} = \sqrt{4} = 2$. Grover, bu elemanı bulmak için yaklaşık $\pi/4 \sqrt{N} \approx 1$ iterasyon gerektirir. (Daha büyük $N$ için bu oran daha hassaslaşır.)

#### Devre Yapısı:

1.  **Başlatma:** Her iki qubite $H$ uygulayarak süperpozisyon oluştur.
2.  **Oracle:** $|11\rangle$ durumuna $-\text{faz}$ uygulayan kapıyı kullan.
3.  **Difüzör:** $H^{\otimes 2}$, $\text{Kontrollü-Z}_{\text{sıfırlara}}$, $H^{\otimes 2}$ uygula.
4.  **Ölçüm:** $q_0$ ve $q_1$'i ölç.

<!-- end list -->

In [1]:
pip install cirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 70.6 MB/s eta 0:00:00


In [2]:
import cirq
import numpy as np

# 1. Qubitleri tanımlayalım (N=4 için 2 Qubit)
q0 = cirq.GridQubit(0, 0)
q1 = cirq.GridQubit(0, 1)

# Aranan durumu ters faz ile işaretleyen ORACLE yapısı
def oracle(q0, q1):
    # Aranan durum |11> (yani kontrol qubitleri 1 ise fazı çevir)
    # Toffoli (CCNOT) kullanarak, hedef qubit |1> olduğunda fazı çevirebiliriz.
    # CNOT'u Kontrollü-Z (CZ) olarak da uygulayabiliriz:
    # Bu, sadece |11> durumuna -faz uygulayan bir devredir (CZ ve H'lerle yapılır).
    # Ancak burada doğrudan |11> fazını çeviren bir yapı kullanalım:

    # 1. Z kapısını Toffoli'nin hedefi gibi kullanmak için Tofoli'yi Z kapısına çevir.
    yield cirq.Z(q0).controlled_by(q1) # Yanlış, iki kontrol lazım

    # Doğru yol: |11> durumuna sadece bir faz çevrimi (Z) uygulamak için
    # Kontrollü-Z kapısını kullanmak ve onu sadece |11> için aktif hale getirmek.
    # Cirq'de CCNOT(Z) = cirq.Z(q0).controlled_by(q1, q2) olarak kullanabiliriz

    # Pratik olarak, Cirq'te CCZ (Kontrollü-Kontrollü-Z) kullanırız:
    yield cirq.CCZ(q0, q1, target=cirq.GridQubit(0, 2)) # CCZ = Kontrollü Z

# Sadece |11> fazını çeviren kapı (CCZ olmadan):
def oracle_simple(q0, q1):
    # X kapıları |0> ları |1> a çevirir. Tofolli(X) |11> de aktif olur.
    # Cirq'de bu:
    yield cirq.CZ(q0, q1) # Sadece |11> fazını çevirir. (CCZ'ye gerek yok)

# Grover Difüzör Kapısı (D)
def diffuser(q0, q1):
    # 1. Tüm q'lara H
    yield cirq.H(q0), cirq.H(q1)

    # 2. Tüm q'lara X (sadece |00> durumunu işaretlemek için)
    yield cirq.X(q0), cirq.X(q1)

    # 3. Kontrollü Z (sadece |11> durumuna -faz uygular, ki bu artık |00|a dönüştü)
    yield cirq.CZ(q0, q1)

    # 4. Tüm q'lara X (geri çevir)
    yield cirq.X(q0), cirq.X(q1)

    # 5. Tüm q'lara H (geri çevir)
    yield cirq.H(q0), cirq.H(q1)


# 1. Başlangıç Devresi: Süperpozisyon
initialization = [
    cirq.H(q0),
    cirq.H(q1),
]

# 2. Grover İterasyonu (Sadece 1 iterasyon yeterli)
grover_iteration = [
    oracle_simple(q0, q1),
    diffuser(q0, q1),
]

# 3. Ölçüm
measurement = [
    cirq.measure(q0, key='q0'),
    cirq.measure(q1, key='q1')
]

circuit = cirq.Circuit(
    initialization,
    grover_iteration * 1, # Sadece 1 iterasyon
    measurement
)

# Simülasyon (1000 kez)
simulator = cirq.Simulator()
results = simulator.run(circuit, repetitions=1000)

print("### Grover Arama Algoritması Devresi (Aranan: |11>) ###")
print(circuit)

print("\n### Ölçüm Sonuçları (1000 Çalıştırma) ###")
# Sonuçlar |q1 q0> formatında okunur. (00=0, 01=1, 10=2, 11=3)
combined_results = results.data.groupby(['q0', 'q1']).size().to_dict()

# Aranan durumun (q0=1, q1=1) yüksek bir olasılıkla bulunması beklenir
print("{Durum (q0 q1): Ölçüm Sayısı}")
for (val0, val1), count in combined_results.items():
    print(f"|{val1}{val0}> : {count}")

### Grover Arama Algoritması Devresi (Aranan: |11>) ###
(0, 0): ───H───@───H───X───@───X───H───M('q0')───
               │           │
(0, 1): ───H───@───H───X───@───X───H───M('q1')───

### Ölçüm Sonuçları (1000 Çalıştırma) ###
{Durum (q0 q1): Ölçüm Sayısı}
|11> : 1000


#### Çıktı Yorumu

Çıktıda, **$|11\rangle$** durumuna karşılık gelen **(q0=1, q1=1)** çiftinin diğer durumların toplamından **çok daha fazla** ölçüldüğünü göreceksiniz. Teorik olarak, 1 iterasyonda bu olasılık %100'e çok yakındır.

Grover algoritmasının gücü, bu tek iterasyon ile (yani sadece bir kez Oracle sorgulama ile), rastgele arama yapmaya göre katlanarak daha hızlı bir şekilde doğru sonuca ulaşmasıdır.
